# 1.0 Import Libraries & Data

In [1]:
import pandas as pd
from extra.utils import load_config

In [2]:
cfg = load_config()

Config loaded successfully!

data:
  raw_path: ../data/raw/smart_manufacturing_data.csv
  interim_path: ../data/interim/cleaned.csv
  processed_path: ../data/processed/features.csv
  target_column: maintenance_required
cleaning: null
features:
  rolling_window_size: 5
  sensor_columns:
  - temperature
  - vibration
  - humidity
  - pressure
  - energy_consumption
model:
  n_jobs: 2
  session_id: 42
  train_size: 0.8
  ignore_features:
  - failure_type
  - downtime_risk
  - anomaly_flag
  - machine_status
  - predicted_remaining_life
mlflow:
  tracking_uri: http://127.0.0.1:5000
  experiment_name: Smart Manufacturing Maintenance



In [3]:
df = pd.read_csv(cfg.data.raw_path)

# 2.0 Data Cleaning

In [4]:
# I know it showed 0 but I will just remove nulls and duplicates just in case.
print(f"Original shape: {df.shape}")
df = df.dropna()
print(f"After removing nulls: {df.shape}")
df = df.drop_duplicates()
print(f"After removing duplicates: {df.shape}")

Original shape: (100000, 13)
After removing nulls: (100000, 13)
After removing duplicates: (100000, 13)


In [5]:
# I will not remove the negative vibrations because it as some correlation with
df[df["vibration"] < 0]["maintenance_required"].value_counts()

maintenance_required
0    31
1     6
Name: count, dtype: int64

In [6]:
# Convert timestamp to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Convert failure_type to categorical (Although this will be dropped for now)
df["failure_type"] = df["failure_type"].astype("category")

In [7]:
df.dtypes

timestamp                   datetime64[ns]
machine_id                           int64
temperature                        float64
vibration                          float64
humidity                           float64
pressure                           float64
energy_consumption                 float64
machine_status                       int64
anomaly_flag                         int64
predicted_remaining_life             int64
failure_type                      category
downtime_risk                      float64
maintenance_required                 int64
dtype: object

In [8]:
# Not dropping here because you can drop later in pycaret's setup()
# df = df.drop(columns=cfg.model.ignore_features)

In [9]:
# Get the last date (without the time)
print(df["timestamp"].max())
last_day = df["timestamp"].dt.normalize().max()
df = df[df["timestamp"].dt.normalize() < last_day]
print(df["timestamp"].max())

2025-03-11 10:39:00
2025-03-10 23:59:00


In [10]:
df.columns

Index(['timestamp', 'machine_id', 'temperature', 'vibration', 'humidity',
       'pressure', 'energy_consumption', 'machine_status', 'anomaly_flag',
       'predicted_remaining_life', 'failure_type', 'downtime_risk',
       'maintenance_required'],
      dtype='object')

In [11]:
df.to_csv(cfg.data.interim_path, index=False)